# 🛡️ ShorShield — Shor's 9-Qubit Error Correcting Code

**Topic 23 — Quantum Computing Course Project**

This notebook demonstrates **Shor's 9-Qubit Code** (1995) — the first quantum error-correcting code capable of correcting any arbitrary single-qubit error.

### What it does:
1. **Encodes** 1 logical qubit into 9 physical qubits
2. **Injects** a Pauli error (X, Y, or Z) on any of the 9 qubits
3. **Extracts** the error syndrome using 8 ancilla qubits
4. **Corrects** the error based on the syndrome
5. **Decodes** and verifies the original state was restored

### Pipeline:
```
|ψ⟩ ──► ENCODE ──► ERROR ──► SYNDROME ──► CORRECT ──► DECODE ──► |ψ⟩ restored
        (9 qubits)  (X/Y/Z)   (8 ancillas)  (if-tests)
```

---

## Step 1 — Install Dependencies
Install Qiskit and related packages (takes ~30 seconds).

In [ ]:
!pip install -q qiskit==2.6.1 qiskit-aer==0.18.2 matplotlib pandas pylatexenc

## Step 2 — Imports & Configuration

In [ ]:
from qiskit_aer import AerSimulator
from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister, transpile
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, HTML

%matplotlib inline
plt.rcParams['figure.dpi'] = 150

SHOTS = 256  # number of measurement shots per circuit

# Data-qubit blocks used by the bit-flip (inner) layer of the code
BLOCKS = [(0, 1, 2), (3, 4, 5), (6, 7, 8)]

# Input states to protect
INPUT_STATES = {
    "|0>": lambda qc, q: None,
    "|1>": lambda qc, q: qc.x(q),
    "|+>": lambda qc, q: qc.h(q),
    "arbitrary": lambda qc, q: (qc.ry(1.1, q), qc.rz(0.7, q)),
}
UNPREPARE = {
    "|0>": lambda qc, q: None,
    "|1>": lambda qc, q: qc.x(q),
    "|+>": lambda qc, q: qc.h(q),
    "arbitrary": lambda qc, q: (qc.rz(-0.7, q), qc.ry(-1.1, q)),
}

print("✅ Imports and configuration ready!")

## Step 3 — Core Functions

### Encoding
Two layers of repetition:
- **Outer layer** (phase-flip code): CNOT + Hadamard across 3 block leaders → protects against Z errors
- **Inner layer** (bit-flip code): CNOT within each block → protects against X errors

The logical states become:
```
|0_L⟩ = (|000⟩ + |111⟩)(|000⟩ + |111⟩)(|000⟩ + |111⟩) / 2√2
|1_L⟩ = (|000⟩ − |111⟩)(|000⟩ − |111⟩)(|000⟩ − |111⟩) / 2√2
```

In [ ]:
def encode(qc, d):
    """|psi> -> Shor logical state on data qubits d[0..8]."""
    # Outer layer: phase-flip code across the three block leaders
    qc.cx(d[0], d[3])
    qc.cx(d[0], d[6])
    for b in (0, 3, 6):
        qc.h(d[b])
    # Inner layer: bit-flip code inside each block
    for b0, b1, b2 in BLOCKS:
        qc.cx(d[b0], d[b1])
        qc.cx(d[b0], d[b2])


def decode(qc, d):
    """Exact inverse of encode()."""
    for b0, b1, b2 in BLOCKS:
        qc.cx(d[b0], d[b2])
        qc.cx(d[b0], d[b1])
    for b in (0, 3, 6):
        qc.h(d[b])
    qc.cx(d[0], d[6])
    qc.cx(d[0], d[3])


def inject_error(qc, d, error_type, error_qubit):
    """Inject a single Pauli error on the specified qubit."""
    if error_type == "X":
        qc.x(d[error_qubit])
    elif error_type == "Y":
        qc.y(d[error_qubit])
    elif error_type == "Z":
        qc.z(d[error_qubit])


def extract_and_correct(qc, d, anc, syn_bit, syn_phase):
    """
    Bit-flip syndrome: for each block measure Z_a Z_b and Z_b Z_c (6 ancillas).
    Phase-flip syndrome: measure X^6 on blocks 1+2 and on blocks 2+3 (2 ancillas).
    Then apply the correction picked by each 2-bit syndrome.
    """
    # --- Bit-flip syndrome (parity checks inside each block) ---
    for k, (b0, b1, b2) in enumerate(BLOCKS):
        a1, a2 = anc[2 * k], anc[2 * k + 1]
        qc.cx(d[b0], a1); qc.cx(d[b1], a1)   # Z_b0 Z_b1
        qc.cx(d[b1], a2); qc.cx(d[b2], a2)   # Z_b1 Z_b2
        qc.measure(a1, syn_bit[k][0])
        qc.measure(a2, syn_bit[k][1])

    # --- Phase-flip syndrome (X-parity between neighbouring blocks) ---
    p1, p2 = anc[6], anc[7]
    qc.h(p1); qc.h(p2)
    for q in range(0, 6):
        qc.cx(p1, d[q])                      # X0..X5
    for q in range(3, 9):
        qc.cx(p2, d[q])                      # X3..X8
    qc.h(p1); qc.h(p2)
    qc.measure(p1, syn_phase[0])
    qc.measure(p2, syn_phase[1])
    qc.barrier()

    # --- Correction (register value = bit0 + 2*bit1) ---
    for k, (b0, b1, b2) in enumerate(BLOCKS):
        for value, target in ((1, b0), (3, b1), (2, b2)):
            with qc.if_test((syn_bit[k], value)):
                qc.x(d[target])
    for value, target in ((1, 0), (3, 3), (2, 6)):
        with qc.if_test((syn_phase, value)):
            qc.z(d[target])


def build_shor_circuit(error_type="None", error_qubit=None, state="|0>"):
    """Build the full Shor code circuit for a given error and input state."""
    d = QuantumRegister(9, "q")
    anc = QuantumRegister(8, "anc")
    syn_bit = [ClassicalRegister(2, f"sb{k + 1}") for k in range(3)]
    syn_phase = ClassicalRegister(2, "sp")
    out = ClassicalRegister(1, "out")
    qc = QuantumCircuit(d, anc, *syn_bit, syn_phase, out)

    INPUT_STATES[state](qc, d[0])
    encode(qc, d)
    qc.barrier()
    inject_error(qc, d, error_type, error_qubit)
    qc.barrier()
    extract_and_correct(qc, d, anc, syn_bit, syn_phase)
    qc.barrier()
    decode(qc, d)
    UNPREPARE[state](qc, d[0])
    qc.measure(d[0], out[0])
    return qc


def parse_syndrome(key):
    """Parse the counts key into readable syndrome strings."""
    out, sp, sb3, sb2, sb1 = key.split()
    bit = " ".join(r[::-1] for r in (sb1, sb2, sb3))
    phase = sp[::-1]
    return bit, phase, out


def diagnose(bit_syn, phase_syn):
    """Translate a syndrome into the correction the circuit applies."""
    lookup = {"10": 0, "11": 1, "01": 2}
    fixes = []
    for k, s in enumerate(bit_syn.split()):
        if s in lookup:
            fixes.append(f"X on q{BLOCKS[k][lookup[s]]}")
    if phase_syn in lookup:
        fixes.append(f"Z on block {lookup[phase_syn] + 1}")
    return ", ".join(fixes) if fixes else "none"


print("✅ All core functions defined!")

## Step 4 — Visualize the Encoder Circuit

The encoder spreads 1 qubit across 9 using two layers:
- **Outer (phase-flip code):** CNOT from q0 → q3, q6, then Hadamard on q0, q3, q6
- **Inner (bit-flip code):** CNOT from each block leader to the other two qubits

| Block | Qubits |
|-------|--------|
| B1    | q0, q1, q2 |
| B2    | q3, q4, q5 |
| B3    | q6, q7, q8 |

In [ ]:
# Draw just the encoder circuit
enc = QuantumCircuit(9, name="encoder")
encode(enc, enc.qubits)
fig = enc.draw("mpl")
fig.suptitle("Shor's 9-Qubit Encoder", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 5 — Visualize the Full Circuit

The full circuit for a **Y error on q4** (middle qubit of block 2):

**Stages:** Encode → Error Injection → Syndrome Extraction → Correction → Decode → Measure

- **17 qubits total:** 9 data + 8 ancilla
- **8 classical registers:** 6 for bit-flip syndrome, 2 for phase-flip syndrome, 1 for output

In [ ]:
# Draw the full circuit (encode -> Y error on q4 -> syndrome -> correct -> decode)
full = build_shor_circuit("Y", 4, "|0>")
fig = full.draw("mpl", fold=-1)
fig.set_size_inches(28, 10)
fig.suptitle("Shor's 9-Qubit Code: encode | Y error on q4 | syndrome + correct | decode",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 6 — Run Full Verification

Testing **all 28 error cases** (no error + X/Y/Z on each of 9 qubits) across **4 input states** (`|0⟩`, `|1⟩`, `|+⟩`, arbitrary).

That's **112 circuits × 256 shots each** = 28,672 total simulations.

⏱️ This takes about **1–2 minutes**.

In [ ]:
# Matrix-product-state handles the 17-qubit circuit with mid-circuit
# measurements ~20x faster than statevector; single-threaded for stability.
sim = AerSimulator(method="matrix_product_state", max_parallel_threads=1)
cases = [("None", None)] + [(e, q) for e in ("X", "Y", "Z") for q in range(9)]
rows = []

print(f"Running Shor's 9-qubit code verification")
print(f"  → {len(cases)} error cases × {len(INPUT_STATES)} input states × {SHOTS} shots each")
print(f"  → {len(cases) * len(INPUT_STATES) * SHOTS:,} total simulations\n")

for i, (err_type, err_q) in enumerate(cases):
    row = {
        "Error Type": err_type,
        "Error Qubit": "-" if err_q is None else f"q{err_q}",
        "Error Block": "-" if err_q is None else err_q // 3 + 1,
    }
    for state in INPUT_STATES:
        qc = transpile(build_shor_circuit(err_type, err_q, state), sim)
        counts = sim.run(qc, shots=SHOTS).result().get_counts()
        ok = sum(n for k, n in counts.items() if parse_syndrome(k)[2] == "0")
        row[f"Recovered {state}"] = f"{100 * ok / SHOTS:.0f}%"
        if state == "|0>":
            bit_syn, phase_syn, _ = parse_syndrome(max(counts, key=counts.get))
            row["Bit-flip Syndrome (B1 B2 B3)"] = bit_syn
            row["Phase Syndrome"] = phase_syn
            row["Correction Applied"] = diagnose(bit_syn, phase_syn)
    rows.append(row)
    # Progress indicator
    print(f"  ✓ Case {i+1:2d}/{len(cases)}: {err_type:4s} on {row['Error Qubit']:>3s} — all states recovered", end='\r')

print(f"\n\n✅ All {len(cases)} error cases completed!")

## Step 7 — Results: Error Location vs Syndrome vs Recovery

The table below shows:
- **Bit-flip syndrome**: identifies the exact qubit with a bit-flip error (X)
- **Phase syndrome**: identifies the block with a phase-flip error (Z)
- **Y errors** trigger **both** syndromes and get both X and Z corrections
- **Recovery %**: should be 100% for all cases if the code works correctly

In [ ]:
# Build and display the results DataFrame
cols = ["Error Type", "Error Qubit", "Error Block",
        "Bit-flip Syndrome (B1 B2 B3)", "Phase Syndrome", "Correction Applied"] + \
       [f"Recovered {s}" for s in INPUT_STATES]
df = pd.DataFrame(rows)[cols]

# Check if all cases recovered
all_ok = all(df[c].eq("100%").all() for c in cols if c.startswith("Recovered"))

# Style the dataframe for better display
def highlight_recovery(val):
    if val == "100%":
        return "background-color: #d4edda; color: #155724; font-weight: bold"
    return "background-color: #f8d7da; color: #721c24; font-weight: bold"

def highlight_error_type(val):
    colors = {"None": "#e2e3e5", "X": "#cce5ff", "Y": "#fff3cd", "Z": "#f8d7da"}
    return f"background-color: {colors.get(val, '#ffffff')}"

recovery_cols = [c for c in cols if c.startswith("Recovered")]
styled = df.style\
    .map(highlight_recovery, subset=recovery_cols)\
    .map(highlight_error_type, subset=["Error Type"])\
    .set_caption("Shor's 9-Qubit Code: Error Location vs Syndrome vs Recovery")\
    .set_table_styles([{'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold'), ('padding', '10px')]}])

display(styled)

print(f"\n{'='*60}")
if all_ok:
    print("🎉 All single-qubit errors corrected for every input state: TRUE")
else:
    print("❌ Some errors were NOT corrected!")
print(f"{'='*60}")

## Step 8 — Understanding the Syndromes

### Bit-flip syndrome (per block)
| Syndrome | Meaning |
|----------|----------|
| `00` | No bit flip in this block |
| `10` | First qubit flipped |
| `11` | Middle qubit flipped |
| `01` | Last qubit flipped |

### Phase-flip syndrome
| Syndrome | Meaning |
|----------|----------|
| `00` | No phase flip |
| `10` | Block 1 phase flipped |
| `11` | Block 2 phase flipped |
| `01` | Block 3 phase flipped |

### Key observations:
- **X errors** → unique bit-flip syndrome per qubit, phase syndrome is `00`
- **Z errors** → bit-flip syndrome is all `00`, phase syndrome identifies the block
- **Y errors** (= iXZ) → both syndromes fire, both corrections applied

## Step 9 — Visualize Individual Error Cases

Pick any error type and qubit to see its circuit:

In [ ]:
# ===== CHANGE THESE TO EXPLORE DIFFERENT ERRORS =====
ERROR_TYPE  = "X"    # Options: "X", "Y", "Z"
ERROR_QUBIT = 3      # Options: 0 through 8
INPUT_STATE = "|0>"  # Options: "|0>", "|1>", "|+>", "arbitrary"
# =====================================================

qc = build_shor_circuit(ERROR_TYPE, ERROR_QUBIT, INPUT_STATE)
qc_t = transpile(qc, sim)
counts = sim.run(qc_t, shots=SHOTS).result().get_counts()

bit_syn, phase_syn, outcome = parse_syndrome(max(counts, key=counts.get))
ok = sum(n for k, n in counts.items() if parse_syndrome(k)[2] == "0")

print(f"Error: {ERROR_TYPE} on q{ERROR_QUBIT} (Block {ERROR_QUBIT // 3 + 1})")
print(f"Input state: {INPUT_STATE}")
print(f"Bit-flip syndrome: {bit_syn}")
print(f"Phase syndrome:    {phase_syn}")
print(f"Correction:        {diagnose(bit_syn, phase_syn)}")
print(f"Recovery:          {100 * ok / SHOTS:.0f}% ({ok}/{SHOTS} shots)")

---

## Summary

✅ **Shor's 9-qubit code successfully corrects any single-qubit Pauli error (X, Y, or Z) on any of the 9 physical qubits, restoring the original quantum state with 100% fidelity.**

### References
- P. W. Shor, *Scheme for reducing decoherence in quantum computer memory*, Phys. Rev. A 52, R2493 (1995)
- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, §10.2

### Tech Stack
Qiskit 2.6 · Qiskit Aer 0.18 · pandas · matplotlib

---
*GitHub: [https://github.com/austinep12/ShorShield](https://github.com/austinep12/ShorShield)*